In [ ]:
# Cell 1: Environment Diagnostics & Restore Target Identification
import os, sys, shutil, hashlib, json, time
from pathlib import Path

print('=== RESTORE-ONLY VERIFICATION: santhosh939s/rebuiltv4-training-checkpoints ===')
print(f'Python version: {sys.version}')

# Verify PyTorch and Transformers presence in fresh Kaggle session
import torch, transformers, peft
print(f'PyTorch: {torch.__version__} (CUDA available: {torch.cuda.is_available()})')
print(f'Transformers: {transformers.__version__}')
print(f'PEFT: {peft.__version__}')

# Identify mounted dataset path
KAGGLE_INPUT_DIR = Path('/kaggle/input/rebuiltv4-training-checkpoints')
RESTORE_TMP_DIR = Path('/tmp/restore_verification_fresh')

if RESTORE_TMP_DIR.exists():
    shutil.rmtree(RESTORE_TMP_DIR)
RESTORE_TMP_DIR.mkdir(parents=True, exist_ok=True)

print(f'Mounted Dataset Exists: {KAGGLE_INPUT_DIR.exists()}')
if KAGGLE_INPUT_DIR.exists():
    print('Input directory tree:')
    for item in sorted(KAGGLE_INPUT_DIR.rglob('*')):
        if item.is_file():
            rel = item.relative_to(KAGGLE_INPUT_DIR)
            print(f'  {str(rel):<50} ({item.stat().st_size:,} bytes)')
            # Copy to fresh restore temporary directory for independent validation
            dest = RESTORE_TMP_DIR / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, dest)
    print(f'Successfully copied Version 1 files to {RESTORE_TMP_DIR}')
else:
    raise FileNotFoundError(f'Target dataset not mounted at {KAGGLE_INPUT_DIR}')


In [ ]:
# Cell 2: Cryptographic SHA256 Checksum & File Inventory Verification
checksum_file = RESTORE_TMP_DIR / 'manifest_checksums.sha256'
manifest_file = RESTORE_TMP_DIR / 'CHECKPOINT_MANIFEST.json'

assert checksum_file.exists(), 'manifest_checksums.sha256 missing from restored dataset!'
assert manifest_file.exists(), 'CHECKPOINT_MANIFEST.json missing from restored dataset!'

with open(manifest_file, 'r', encoding='utf-8') as f:
    manifest_data = json.load(f)

print(f"Manifest Dataset ID: {manifest_data.get('dataset_id')}")
print(f"Source Archive: {manifest_data.get('source_archive')}")
print(f"Total Expected Checkpoint Files: {manifest_data.get('total_staged_files')}")

checksum_results = {}
all_checksums_pass = True

with open(checksum_file, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        exp_sha, rel_path = line.split(maxsplit=1)
        target = RESTORE_TMP_DIR / rel_path
        if not target.exists():
            print(f'MISSING FILE: {rel_path}')
            all_checksums_pass = False
            checksum_results[rel_path] = {'status': 'MISSING', 'expected': exp_sha}
            continue
        
        h = hashlib.sha256()
        with open(target, 'rb') as tf:
            while chunk := tf.read(1024*1024):
                h.update(chunk)
        act_sha = h.hexdigest()
        match = (act_sha == exp_sha)
        if not match:
            all_checksums_pass = False
            print(f'HASH MISMATCH: {rel_path}')
        checksum_results[rel_path] = {
            'status': 'MATCH' if match else 'MISMATCH',
            'sha256': act_sha,
            'size_bytes': target.stat().st_size
        }

print(f'Verified {len(checksum_results)} files against manifest: ALL_CHECKSUMS_MATCH = {all_checksums_pass}')
assert all_checksums_pass, 'Cryptographic checksum verification failed!'


In [ ]:
# Cell 3: Verification of Required Checkpoint Components for Resumption
REQUIRED_COMPONENTS = [
    'adapter_model.safetensors',
    'adapter_config.json',
    'optimizer.pt',
    'scheduler.pt',
    'scaler.pt',
    'trainer_state.json',
    'training_args.bin',
    'rng_state_0.pth',
    'rng_state_1.pth',
    'tokenizer.json',
    'tokenizer_config.json',
    'chat_template.jinja'
]

checkpoints_audit = {}
for ckpt in ['checkpoint-200', 'checkpoint-300']:
    ckpt_dir = RESTORE_TMP_DIR / ckpt
    assert ckpt_dir.exists(), f'{ckpt} directory missing!'
    missing = []
    for comp in REQUIRED_COMPONENTS:
        comp_file = ckpt_dir / comp
        if not comp_file.exists():
            missing.append(comp)
    checkpoints_audit[ckpt] = {
        'exists': True,
        'all_components_present': len(missing) == 0,
        'missing_components': missing,
        'total_files': len(list(ckpt_dir.iterdir()))
    }
    print(f'[{ckpt}] Component check: Missing: {missing} (Total files: {checkpoints_audit[ckpt]["total_files"]})')
    assert len(missing) == 0, f'{ckpt} missing critical resumption components: {missing}'


In [ ]:
# Cell 4: Base Model & Training Dataset Resolution Verification
print('Verifying base model and dataset reference resolution...')

# 1. Base model resolution without downloading 3GB weights
from transformers import AutoConfig, AutoTokenizer

base_model_ref = 'Qwen/Qwen2.5-1.5B-Instruct'
print(f'Querying remote Hugging Face registry for base model: {base_model_ref}...')
hf_config = AutoConfig.from_pretrained(base_model_ref)
print(f'  Base model resolved successfully: {hf_config.model_type}, hidden_size={hf_config.hidden_size}, num_layers={hf_config.num_hidden_layers}')

# 2. Local restored tokenizer test
local_tok_200 = AutoTokenizer.from_pretrained(str(RESTORE_TMP_DIR / 'checkpoint-200'))
print(f'  Checkpoint-200 Tokenizer loaded: vocab_size={len(local_tok_200)}')
test_tokens = local_tok_200.encode('GET /api/v1/user?id=1 HTTP/1.1')
print(f'  Tokenizer encoding verified: {len(test_tokens)} tokens generated.')

# 3. Dataset references resolution
# Verify test dataset exists in recovery
rec_data_path = Path('/kaggle/input/recovery')
test_sample_path = None
if rec_data_path.exists():
    for p in rec_data_path.rglob('test_sample_2000.jsonl'):
        test_sample_path = p
        break
print(f'Held-out test set resolved in recovery dataset: {test_sample_path}')

# Verify training script syntax and references
train_script = RESTORE_TMP_DIR / 'training_configuration' / 'train_rebuilt_v4.py'
assert train_script.exists(), 'train_rebuilt_v4.py missing!'
with open(train_script, 'r', encoding='utf-8') as f:
    script_content = f.read()
assert 'Qwen/Qwen2.5-1.5B-Instruct' in script_content, 'Base model string mismatch in training script!'
print('Training script syntax and references verified.')


In [ ]:
# Cell 5: Deep Checkpoint State Loading & Resume Step Identification
print('Performing deep deserialization of PyTorch optimizer and trainer states...')

resumption_validation = {}

for ckpt in ['checkpoint-200', 'checkpoint-300']:
    ckpt_path = RESTORE_TMP_DIR / ckpt
    print(f'\n--- Testing Checkpoint State Loading: {ckpt} ---')
    
    # 1. Load trainer_state.json
    with open(ckpt_path / 'trainer_state.json', 'r') as f:
        ts = json.load(f)
    resume_step = ts.get('global_step')
    epoch = ts.get('epoch')
    last_log = ts.get('log_history', [])[-1] if ts.get('log_history') else {}
    print(f'  [TrainerState] Identified Global Step: {resume_step}, Epoch: {epoch:.4f}')
    print(f'  [TrainerState] Last Metric Log: {last_log}')
    
    # 2. Deserializing PyTorch Optimizer State
    t0 = time.time()
    opt_state = torch.load(ckpt_path / 'optimizer.pt', map_location='cpu')
    t_opt = time.time() - t0
    param_groups = opt_state.get('param_groups', [])
    state_dict = opt_state.get('state', {})
    print(f'  [Optimizer] Deserialized in {t_opt:.2f}s | Param Groups: {len(param_groups)} | Tracked Tensors: {len(state_dict)}')
    lr = param_groups[0].get('lr') if param_groups else None
    print(f'  [Optimizer] Preserved Momentum Learning Rate: {lr}')
    
    # 3. Deserializing Scheduler State
    sched_state = torch.load(ckpt_path / 'scheduler.pt', map_location='cpu')
    print(f'  [Scheduler] Deserialized successfully: last_epoch={sched_state.get("last_epoch")}')
    
    # 4. Deserializing GradScaler State
    scaler_state = torch.load(ckpt_path / 'scaler.pt', map_location='cpu')
    print(f'  [GradScaler] Deserialized successfully: scale={scaler_state.get("scale")}')
    
    # 5. Deserializing RNG States (weights_only=False required for PyTorch 2.6+ numpy state)
    rng_0 = torch.load(ckpt_path / 'rng_state_0.pth', map_location='cpu', weights_only=False)
    rng_1 = torch.load(ckpt_path / 'rng_state_1.pth', map_location='cpu', weights_only=False)
    print(f'  [RNG States] Deserialized successfully: RNG 0 keys={list(rng_0.keys()) if isinstance(rng_0, dict) else type(rng_0)}, RNG 1 keys={list(rng_1.keys()) if isinstance(rng_1, dict) else type(rng_1)}')
    
    # 6. Deserializing Training Arguments (TrainingArguments object requires weights_only=False)
    training_args = torch.load(ckpt_path / 'training_args.bin', weights_only=False)
    out_dir = getattr(training_args, 'output_dir', str(training_args))
    m_steps = getattr(training_args, 'max_steps', None)
    print(f'  [TrainingArgs] Deserialized successfully: Output Dir={out_dir}, Max Steps={m_steps}')
    
    resumption_validation[ckpt] = {
        'identified_resume_step': resume_step,
        'epoch': epoch,
        'learning_rate': lr,
        'param_groups_count': len(param_groups),
        'optimizer_tensors_count': len(state_dict),
        'deserialization_status': 'SUCCESS'
    }

print('\nAll checkpoint states successfully deserialized and verified.')


In [ ]:
# Cell 6: Generate Formal Restoration Report & Verification Artifact
report_path = Path('/kaggle/working/restore_verification_report.json')

final_report = {
    'task': 'restore_only_verification',
    'target_dataset': 'santhosh939s/rebuiltv4-training-checkpoints',
    'dataset_version': 1,
    'verification_timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'overall_status': 'PASS',
    'execution_environment': {
        'python': sys.version.split()[0],
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'peft': peft.__version__,
        'cuda_available': torch.cuda.is_available()
    },
    'checksums_verified_count': len(checksum_results),
    'all_checksums_match': all_checksums_pass,
    'checkpoints_audit': checkpoints_audit,
    'base_model_reference_resolved': {
        'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
        'resolved': True,
        'architecture': hf_config.model_type
    },
    'resumption_validation': resumption_validation,
    'optimizer_steps_performed': 0,
    'model_weights_downloaded_from_hf': False
}

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(final_report, f, indent=2)

print('===================================================================')
print('RESTORE VERIFICATION RESULT: PASS')
print(f'Report written to: {report_path} ({report_path.stat().st_size:,} bytes)')
print('===================================================================')
